# Literate Science Results: sexual dimorphism in Palmer penguins

```{note} Literate science
The text, numbers and figures on this page come from the project's code and data, and are rebuilt every time the site is published. [How this page is made](#how-this-page-is-made).
```

How much heavier is a male penguin than a female of the same species, and can a simple model tell a penguin's recorded sex from its measurements? This page answers both with the [penguin project](../project.md)'s own code, on all three field seasons, 2007 to 2009. Data: palmerpenguins (Horst, Hill and Gorman 2020, [doi:10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218)), CC0, from Gorman, Williams and Fraser (2014), PLOS ONE 9(3): e90081. Collected by Palmer Station LTER with NSF support.

## Run the project

- This copies the finished project, Exercise 4's files and its data, into a temporary folder and runs its scripts there, unchanged.

In [ ]:
import json
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import pandas as pd
import yaml

EXERCISE = Path("../exercises/04-registry/.exercise")
project = Path(tempfile.mkdtemp()) / "project"
shutil.copytree(EXERCISE / "project", project)
(project / "data").mkdir()
shutil.copyfile(EXERCISE / "data" / "penguins.csv", project / "data" / "penguins.csv")


def run(script):
    done = subprocess.run([sys.executable, script], cwd=project, check=True, capture_output=True, text=True)
    print(f"$ python {script}\n{done.stdout}", end="")


for script in ("clean.py", "validate.py", "report.py"):
    run(script)

## Body mass

- Male mean body mass minus female mean body mass, per species, from `metrics.json`, with the number of penguins in each group.

In [ ]:
metrics = json.loads((project / "metrics.json").read_text())
penguins = pd.read_csv(project / "results" / "validated.csv")
counts = penguins.groupby(["species", "sex"]).size().unstack()
pd.DataFrame({
    "Male minus female (g)": [metrics[f"dimorphism_g_{species.lower()}"] for species in counts.index],
    "Males": counts["male"],
    "Females": counts["female"],
}, index=counts.index)

In [ ]:
import matplotlib.pyplot as plt

groups = penguins.groupby(["species", "sex"])["body_mass_g"]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.boxplot([mass for _, mass in groups], tick_labels=[f"{species}\n{sex}" for species, sex in groups.groups])
ax.set_ylabel("Body mass (g)")
ax.set_title("Body mass by species and sex, 2007 to 2009")
plt.show()

## Predicting recorded sex

- Logistic regression on bill length, bill depth, flipper length, body mass and species, trained on 2007 and 2008 and validated on 2009.
- It runs `split.py` and `train.py` at `C: 0.01` and at `C: 1`, the two versions Exercise 4 registers.

In [ ]:
accuracy = {}
for C in (0.01, 1):
    params = yaml.safe_load((project / "params.yaml").read_text())
    params["train"]["C"] = C
    (project / "params.yaml").write_text(yaml.safe_dump(params))
    run("split.py")
    run("train.py")
    accuracy[C] = json.loads((project / "model_metrics.json").read_text())["val_accuracy"]

splits = json.loads((project / "splits.json").read_text())
print(f"\nTraining rows: {len(splits['train'])}. Validation rows: {len(splits['val'])}.")
for C, value in accuracy.items():
    print(f"Validation accuracy at C = {C}: {value}")

- The recorded sex came from a DNA test. The model is a classroom example checked against that label, not a replacement for it.

## Methods

- The scripts are `clean.py`, `validate.py`, `report.py`, `split.py` and `train.py`, with `params.yaml`, from `exercises/04-registry/.exercise/project/`. The data is `exercises/04-registry/.exercise/data/penguins.csv`.
- `clean.py` drops penguins with no body mass or no sex. `validate.py` checks the columns and that rows remain. `report.py` takes the mean body mass by species and sex and subtracts.
- `split.py` puts 2007 and 2008 in training and 2009 in validation. `train.py` standardizes the four measurements and one-hot species columns, and fits scikit-learn's `LogisticRegression` with `max_iter=1000`.
- The packages come from `results/environment.yml`. These are the versions this build used:

In [ ]:
import platform

import matplotlib
import sklearn

print(f"Python {platform.python_version()}, pandas {pd.__version__}, scikit-learn {sklearn.__version__}, "
      f"matplotlib {matplotlib.__version__}, PyYAML {yaml.__version__}")

## How this page is made

- The site's build runs this notebook on every publish, in its own environment from `results/environment.yml`, with `--strict`.
- So every number and the figure above come from that run. If the code breaks, the build fails and the page isn't published.
- [How this site is built](../build.md) has the configuration and the commands.